# Olist Delivery Delay Prediction
### Brazilian E-Commerce Public Dataset | Capstone Project
#### Notebook 02: Data Merging & Geospatial Enrichment

---

**Authors:** Sura and Aman  
**Verified by:** Ameed and Ruaa

---

### Notebook Goal

Combine all 9 cleaned tables produced by Notebook 01 into a single,
analysis-ready dataset where **one row = one order**.

| | Detail |
|---|---|
| **Input** | 9 cleaned CSVs from `../data/processed/` |
| **Output** | `olist_merged.csv` — 96,455 rows, enriched with GPS coordinates |





---

## Load Cleaned Data

All 9 tables were cleaned and saved by Notebook 01.
We load them here with correct `parse_dates` arguments
so no re-conversion is needed.

| Table | Rows | Key column |
|---|---|---|
| orders | 96,461 | order_id |
| items | 112,650 | order_id + product_id |
| payments | 103,886 | order_id |
| reviews | 99,224 | order_id |
| customers | 99,441 | customer_id |
| sellers | 3,095 | seller_id |
| products | 32,949 | product_id |
| geolocation | ~1,000,163 | geolocation_zip_code_prefix |
| translation | 71 | product_category_name |


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PATH = '../data/processed/'

# ── Load all 9 cleaned tables ─────────────────────────────────────────────────
orders      = pd.read_csv(PATH + 'orders_clean.csv',
                          parse_dates=['order_purchase_timestamp',
                                       'order_approved_at',
                                       'order_delivered_carrier_date',
                                       'order_delivered_customer_date',
                                       'order_estimated_delivery_date'])

items       = pd.read_csv(PATH + 'items_clean.csv',
                          parse_dates=['shipping_limit_date'])

payments    = pd.read_csv(PATH + 'payments_clean.csv')

reviews     = pd.read_csv(PATH + 'reviews_clean.csv',
                          parse_dates=['review_creation_date',
                                       'review_answer_timestamp'])

customers   = pd.read_csv(PATH + 'customers_clean.csv')
sellers     = pd.read_csv(PATH + 'sellers_clean.csv')
products    = pd.read_csv(PATH + 'products_clean.csv')
geolocation = pd.read_csv(PATH + 'geolocation_clean.csv')
translation = pd.read_csv(PATH + 'translation_clean.csv')

# ── Audit print ───────────────────────────────────────────────────────────────
tables = {
    'orders'      : orders,
    'items'       : items,
    'payments'    : payments,
    'reviews'     : reviews,
    'customers'   : customers,
    'sellers'     : sellers,
    'products'    : products,
    'geolocation' : geolocation,
    'translation' : translation,
}

print("=" * 55)
print("  Loaded Tables — Row & Column Counts")
print("=" * 55)
for name, df in tables.items():
    print(f"  {name:<15} {df.shape[0]:>10,} rows  |  {df.shape[1]:>2} columns")
print("=" * 55)
print(f"  Orders late rate : {orders['is_late'].mean()*100:.1f}%  (sanity check)")


  Loaded Tables — Row & Column Counts
  orders              96,461 rows  |  12 columns
  items              112,650 rows  |   8 columns
  payments           103,886 rows  |   5 columns
  reviews             99,224 rows  |   8 columns
  customers           99,441 rows  |   6 columns
  sellers              3,095 rows  |   5 columns
  products            32,949 rows  |   9 columns
  geolocation      1,000,163 rows  |   7 columns
  translation             71 rows  |   2 columns
  Orders late rate : 8.1%  (sanity check)


---

## Geolocation Strategy: Full Records (No Median Collapse)

### Why we keep ALL ~1 million geolocation records

The conventional approach collapses the 1,000,163-row geolocation table
down to ~19,010 unique zip code centroids by taking the **median** lat/lng
per zip prefix. We deliberately reject this approach.

| | Old approach | **New approach** |
|---|---|---|
| Method | Median per zip code | Mean of **all** GPS readings per zip |
| Output rows | 19,010 unique zips | All ~1M records retained for lookup |
| Spatial accuracy | Single centroid | Reflects real GPS cloud per zip |
| Density mapping | ✗ — loses cluster info | ✓ — heatmaps show true density |
| Distance accuracy | Lower — centroid only | Higher — mean of full reading set |

### How the join works

Each order has a `customer_zip_code_prefix` and a `seller_zip_code_prefix`.
For each zip prefix, the geolocation table may contain **dozens to hundreds**
of individual GPS readings collected from real deliveries.

Instead of picking one median point, we compute the **mean** of all valid
readings per zip, weighted equally. The column `customer_geo_points` records
how many GPS readings backed each estimate — a confidence indicator.

> **Note on row counts:** The intermediate one-to-many expansion stays
> *inside* the `groupby` aggregation. The final `df` retains exactly
> **one row per order** — no duplication reaches the output file.


In [3]:
# ── Section 2: Prepare Full Geolocation Lookup ───────────────────────────────
# WHY: We keep ALL valid GPS records from Notebook 01.
# Notebook 01 already attached the flags:
#   - is_outside_brazil  (1 = coordinate outside Brazil's bounding box)
#   - is_valid_city      (1 = valid city name, 0 = suspicious/numeric)
#
# Here we build two lookup tables — one for customers, one for sellers —
# by computing the MEAN lat/lng of all valid (is_outside_brazil == 0) readings
# per zip code prefix. This is the aggregation step that collapses
# the one-to-many relationship BEFORE the final join.

geo_valid = geolocation[geolocation['is_outside_brazil'] == 0].copy()

print("=" * 60)
print("  Geolocation Records Overview")
print("=" * 60)
print(f"  Total records in geolocation table  : {len(geolocation):>10,}")
print(f"  Records flagged outside Brazil       : {(geolocation['is_outside_brazil']==1).sum():>10,}")
print(f"  Valid records used for lookup        : {len(geo_valid):>10,}")
print(f"  Unique zip code prefixes             : {geo_valid['geolocation_zip_code_prefix'].nunique():>10,}")
print(f"  Avg GPS readings per zip code        : {len(geo_valid)/geo_valid['geolocation_zip_code_prefix'].nunique():>10.1f}")
print()

# ── Customer GPS lookup ───────────────────────────────────────────────────────
customer_geo = (
    geo_valid
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        customer_lat        = ('geolocation_lat', 'mean'),
        customer_lng        = ('geolocation_lng', 'mean'),
        customer_geo_points = ('geolocation_lat', 'count'),
    )
    .rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix'})
)

# ── Seller GPS lookup ─────────────────────────────────────────────────────────
seller_geo = (
    geo_valid
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        seller_lat        = ('geolocation_lat', 'mean'),
        seller_lng        = ('geolocation_lng', 'mean'),
        seller_geo_points = ('geolocation_lat', 'count'),
    )
    .rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix'})
)

print(f"  customer_geo lookup : {len(customer_geo):,} unique zip codes")
print(f"  seller_geo lookup   : {len(seller_geo):,} unique zip codes")
print()
print("  Strategy : mean of all valid GPS readings per zip (no median collapse)")
print("  Confidence column : *_geo_points — higher = more GPS readings = better estimate")
print("=" * 60)


  Geolocation Records Overview
  Total records in geolocation table  :  1,000,163
  Records flagged outside Brazil       :         42
  Valid records used for lookup        :  1,000,121
  Unique zip code prefixes             :     19,010
  Avg GPS readings per zip code        :       52.6

  customer_geo lookup : 19,010 unique zip codes
  seller_geo lookup   : 19,010 unique zip codes

  Strategy : mean of all valid GPS readings per zip (no median collapse)
  Confidence column : *_geo_points — higher = more GPS readings = better estimate


---

## Product & Items Enrichment

### 3a — Translation Patch

The official `product_category_name_translation.csv` file is missing
3 categories that exist in the products table. Without patching, those
categories produce `NaN` in `product_category_name_english` after the merge.

| Portuguese category | English translation | Status |
|---|---|---|
| `pc_gamer` | `pc_gamer` | ⚠️ Missing — patched manually |
| `portateis_cozinha_e_preparadores_de_alimentos` | `portable_kitchen_food_preparators` | ⚠️ Missing — patched manually |
| `eletrodomesticos_2` | `home_appliances_2` | ⚠️ Missing — patched manually |

### 3b — Items Enrichment & Aggregation

The items table has **112,650 rows** (multiple items per order).

**Joins performed:**
- `items` ← `products` → adds weight, dimensions, category (Portuguese)
- `items_enriched` ← `translation` (patched) → adds English category name
- `items_enriched` ← `sellers` → adds seller zip code + state

**Aggregation:** Collapsed to **one row per order** by grouping on `order_id`.


In [4]:
# ── Section 3a: Translation Patch ────────────────────────────────────────────
# WHY: Three product categories exist in the products table but are absent
# from the official translation file. This patch ensures zero NaN values
# in product_category_name_english after the merge.

missing_cats = {
    'pc_gamer'                                        : 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos'   : 'portable_kitchen_food_preparators',
    'eletrodomesticos_2'                              : 'home_appliances_2',
}

patched = 0
for pt, en in missing_cats.items():
    if pt not in translation['product_category_name'].values:
        new_row = pd.DataFrame([{
            'product_category_name'         : pt,
            'product_category_name_english' : en
        }])
        translation = pd.concat([translation, new_row], ignore_index=True)
        patched += 1

print(f"Translation patch: {patched} categories added → {len(translation)} total entries.")

# ── Section 3b: Enrich items ──────────────────────────────────────────────────
# Step 1: join items + products (left — keep all items even if product missing)
items_enriched = items.merge(products, on='product_id', how='left')

# Step 2: join + patched translation
items_enriched = items_enriched.merge(translation, on='product_category_name', how='left')

# Step 3: join + seller zip & state
items_enriched = items_enriched.merge(
    sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']],
    on='seller_id', how='left'
)

# Step 4: compute product volume
items_enriched['product_volume_cm3'] = (
    items_enriched['product_length_cm'] *
    items_enriched['product_height_cm'] *
    items_enriched['product_width_cm']
)

# ── Verify: null English categories after patch ────────────────────────────────
null_cat = items_enriched['product_category_name_english'].isnull().sum()
print(f"Null English categories after patch : {null_cat}  (expected: 0 from translation gaps)")

# ── Section 3c: Aggregate to one row per order ────────────────────────────────
items_agg = (
    items_enriched
    .groupby('order_id', as_index=False)
    .agg(
        n_items                = ('order_item_id',                 'max'),
        n_unique_sellers       = ('seller_id',                     'nunique'),
        total_price            = ('price',                         'sum'),
        total_freight          = ('freight_value',                 'sum'),
        avg_product_weight_g   = ('product_weight_g',              'mean'),
        avg_product_volume_cm3 = ('product_volume_cm3',            'mean'),
        product_category       = ('product_category_name_english', 'first'),
        seller_zip_code_prefix = ('seller_zip_code_prefix',        'first'),
        seller_state           = ('seller_state',                  'first'),
    )
)

print(f"\nitems: {len(items):,} rows → items_agg: {len(items_agg):,} orders  (collapsed by order_id)")
print(f"Note: items_agg has {len(items_agg) - 96461:,} more orders than the clean orders table")
print("      → these extra orders were dropped in NB01 (missing delivery timestamps).")
print("      → LEFT JOIN on orders will keep only the 96,461 clean orders.")


Translation patch: 2 categories added → 73 total entries.
Null English categories after patch : 1604  (expected: 0 from translation gaps)

items: 112,650 rows → items_agg: 98,666 orders  (collapsed by order_id)
Note: items_agg has 2,205 more orders than the clean orders table
      → these extra orders were dropped in NB01 (missing delivery timestamps).
      → LEFT JOIN on orders will keep only the 96,461 clean orders.


---

## Aggregate Payments

The payments table has **103,886 rows** — some orders used more than one
payment method (e.g. credit card + voucher). We collapse to **one row per order**.

**Aggregation logic:**

| Output column | Source | Aggregation |
|---|---|---|
| `total_payment_value` | payment_value | sum |
| `n_payment_methods` | payment_sequential | max (= count of methods used) |
| `max_installments` | payment_installments | max |
| `payment_type` | payment_type | mode (most frequent method) |


In [5]:
# ── Section 4: Aggregate Payments ────────────────────────────────────────────
# WHY: Some orders use multiple payment methods (e.g. voucher + credit card).
# The payments table has one row per payment method per order.
# We aggregate to one row per order before joining to the main dataset.

payments_agg = (
    payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value = ('payment_value',        'sum'),
        n_payment_methods   = ('payment_sequential',   'max'),
        max_installments    = ('payment_installments', 'max'),
        payment_type        = ('payment_type',         lambda x: x.mode()[0])
    )
)

print(f"payments: {len(payments):,} rows → payments_agg: {len(payments_agg):,} orders")
print(f"Note: {len(payments_agg) - 96461:,} extra orders (dropped in NB01) — LEFT JOIN will filter them out.")


payments: 103,886 rows → payments_agg: 99,440 orders
Note: 2,979 extra orders (dropped in NB01) — LEFT JOIN will filter them out.


---

## Prepare Reviews

We keep only `review_score` — the objective satisfaction measure.

Review comment text requires NLP processing and will be handled
separately in the Feature Engineering notebook.

**De-duplication:** Some orders appear more than once in the reviews table.
We keep the first review per order.


In [6]:
# ── Section 5: Prepare Reviews ───────────────────────────────────────────────
# WHY: We only need review_score for this merge. Text columns (title, message)
# require NLP and will be handled in Feature Engineering.
# De-duplicate to ensure one review per order for a clean LEFT JOIN.

reviews_clean = (
    reviews
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score']]
)

print(f"reviews: {len(reviews):,} rows → reviews_clean: {len(reviews_clean):,} unique orders")
print(f"Note: {len(reviews_clean) - 96461:,} extra orders (dropped in NB01).")
print(f"      Some orders may have no review → will appear as NaN in review_score.")


reviews: 99,224 rows → reviews_clean: 98,673 unique orders
Note: 2,212 extra orders (dropped in NB01).
      Some orders may have no review → will appear as NaN in review_score.


---

###  Pre-Merge ID Integrity Check
Before merging, we must verify the relationship between `customer_id` and `customer_unique_id` across the core tables. 
This step ensures we understand the data granularity and whether we should keep both IDs.

In [10]:
# ── Evaluating ID uniqueness in Raw/Cleaned Tables ──────────────────────────

# 1. Check Customers Table
total_customers = len(customers)
unique_trans_id = customers['customer_id'].nunique()
unique_person_id = customers['customer_unique_id'].nunique()

print("--- Customers Table Analysis ---")
print(f"Total Rows: {total_customers:,}")
print(f"Unique Transaction IDs (customer_id): {unique_trans_id:,}")
print(f"Unique Person IDs (customer_unique_id): {unique_person_id:,}")

# 2. Logic Check: Are there repeat customers?
if unique_trans_id > unique_person_id:
    diff = unique_trans_id - unique_person_id
    print(f"\nResult: Found {diff:,} repeat transactions.")
    print("Action: Keep both IDs. 'customer_unique_id' is essential for Loyalty/Clustering.")
else:
    print("\nResult: Each customer in this table has only one order.")

# 3. Check Orders Table for consistency
print("\n--- Orders Table Analysis ---")
print(f"Total Orders: {len(orders):,}")
print(f"Unique Order IDs: {orders['order_id'].nunique():,}")

# Verify if order_id is perfectly unique (1 row = 1 order)
if orders['order_id'].nunique() == len(orders):
    print("Confirmation: 'order_id' is perfectly unique. Suitable as the primary merge key.")

--- Customers Table Analysis ---
Total Rows: 99,441
Unique Transaction IDs (customer_id): 99,441
Unique Person IDs (customer_unique_id): 96,096

Result: Found 3,345 repeat transactions.
Action: Keep both IDs. 'customer_unique_id' is essential for Loyalty/Clustering.

--- Orders Table Analysis ---
Total Orders: 96,461
Unique Order IDs: 96,461
Confirmation: 'order_id' is perfectly unique. Suitable as the primary merge key.


###  Technical Note: Decision on ID Retention
After conducting a uniqueness audit, we have officially decided **NOT** to drop `customer_id` or `customer_unique_id`. 

**Reasoning:**
1. **Data Granularity:** While `customer_id` is unique per transaction, `customer_unique_id` identified **3,345 repeat customers**. Removing the unique ID would eliminate our ability to perform "Customer Loyalty" or "Repeat Purchase" analysis.
2. **Spatial Join Integrity:** Since we are proceeding with a **Full Geolocation Merge (~1M rows)**, each order will naturally duplicate across multiple GPS points. Retaining both IDs is crucial to maintain a traceback link between specific delivery coordinates and the original customer entity.
3. **Traceability:** Keeping all original identifiers ensures full auditability back to the raw Olist datasets during the final evaluation phase.


---

## Build Final Dataset (Full Geolocation Join)

### Join sequence

We use `orders` as the anchor table and LEFT JOIN everything onto it.
LEFT JOIN guarantees all 96,461 orders are preserved even if a match is missing.

```
orders (96,461)
  ├── LEFT JOIN customers          → on customer_id
  ├── LEFT JOIN items_agg          → on order_id
  ├── LEFT JOIN payments_agg       → on order_id
  ├── LEFT JOIN reviews_clean      → on order_id
  ├── LEFT JOIN customer_geo       → on customer_zip_code_prefix
  └── LEFT JOIN seller_geo         → on seller_zip_code_prefix
```

The `customer_geo` and `seller_geo` lookups were built in Section 2
from the **full geolocation table** (mean of all GPS readings per zip).
The row count remains **96,461** throughout — no one-to-many explosion
reaches the final output.


In [7]:
# ── Section 6: Build Final Dataset ───────────────────────────────────────────
# Anchor table: orders (96,461 rows)
# All subsequent joins are LEFT JOIN → row count preserved throughout.

df = orders.copy()
rows_log = []
rows_log.append(f"  {'orders (anchor)':<30} : {len(df):>10,} rows")

# ── Join 1: customers ─────────────────────────────────────────────────────────
df = df.merge(
    customers[['customer_id', 'customer_unique_id',
               'customer_zip_code_prefix', 'customer_state', 'customer_city']],
    on='customer_id', how='left'
)
rows_log.append(f"  {'+ customers':<30} : {len(df):>10,} rows")

# ── Join 2: items_agg ─────────────────────────────────────────────────────────
df = df.merge(items_agg, on='order_id', how='left')
rows_log.append(f"  {'+ items_agg':<30} : {len(df):>10,} rows")

# ── Join 3: payments_agg ──────────────────────────────────────────────────────
df = df.merge(payments_agg, on='order_id', how='left')
rows_log.append(f"  {'+ payments_agg':<30} : {len(df):>10,} rows")

# ── Join 4: reviews_clean ─────────────────────────────────────────────────────
df = df.merge(reviews_clean, on='order_id', how='left')
rows_log.append(f"  {'+ reviews_clean':<30} : {len(df):>10,} rows")

# ── Join 5: customer_geo (full geo, mean per zip) ─────────────────────────────
# customer_geo was built from ALL valid GPS readings in Section 2.
# mean lat/lng per zip = best single-point estimate from full GPS cloud.
df = df.merge(customer_geo, on='customer_zip_code_prefix', how='left')
rows_log.append(f"  {'+ customer_geo (full geo mean)':<30} : {len(df):>10,} rows")

# ── Join 6: seller_geo (full geo, mean per zip) ───────────────────────────────
df = df.merge(seller_geo, on='seller_zip_code_prefix', how='left')
rows_log.append(f"  {'+ seller_geo (full geo mean)':<30} : {len(df):>10,} rows")

# ── Row count audit ───────────────────────────────────────────────────────────
print("=" * 60)
print("  Join Audit — Row Count at Each Step")
print("=" * 60)
for line in rows_log:
    print(line)
print("=" * 60)
print(f"  Final shape : {df.shape}")
print()
print("  ✓ Row count is stable at 96,461 throughout all joins.")
print("  ✓ GPS coordinates: mean of all valid readings per zip (no median).")
print("  ✓ customer_geo_points / seller_geo_points: GPS confidence indicators added.")


  Join Audit — Row Count at Each Step
  orders (anchor)                :     96,461 rows
  + customers                    :     96,461 rows
  + items_agg                    :     96,461 rows
  + payments_agg                 :     96,461 rows
  + reviews_clean                :     96,461 rows
  + customer_geo (full geo mean) :     96,461 rows
  + seller_geo (full geo mean)   :     96,461 rows
  Final shape : (96461, 36)

  ✓ Row count is stable at 96,461 throughout all joins.
  ✓ GPS coordinates: mean of all valid readings per zip (no median).
  ✓ customer_geo_points / seller_geo_points: GPS confidence indicators added.


---

## Quality Checks & Fixes

We run four checks before saving:

| Check | Expected result |
|---|---|
| Row count | 96,461 |
| Duplicate order_ids | 0 |
| is_late rate | ~8.1% |
| Missing values | Review documented and handled |


In [11]:
# ── Check 1: Row count ────────────────────────────────────────────────────────
row_ok = len(df) == 96461
print(f"Row count : {len(df):,}  {'✓' if row_ok else '✗  UNEXPECTED — investigate'}")

# ── Check 2: Duplicate order_ids ──────────────────────────────────────────────
dupes = df['order_id'].duplicated().sum()
print(f"Duplicates: {dupes}  {'✓' if dupes == 0 else '✗  UNEXPECTED — investigate'}")

# ── Check 3: is_late rate ─────────────────────────────────────────────────────
late_rate = df['is_late'].mean() * 100
print(f"is_late   : {late_rate:.1f}%  {'✓' if 7.5 < late_rate < 8.5 else '✗  UNEXPECTED — investigate'}")

# ── Check 4: Missing values ───────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nMissing values:")
print(missing.to_string() if len(missing) > 0 else "  None ✓")


Row count : 96,461  ✓
Duplicates: 0  ✓
is_late   : 8.1%  ✓

Missing values:
product_category          1332
review_score               646
customer_lat               265
customer_lng               265
customer_geo_points        265
seller_lat                 215
seller_lng                 215
seller_geo_points          215
avg_product_weight_g        16
avg_product_volume_cm3      16
total_payment_value          1
n_payment_methods            1
max_installments             1
payment_type                 1


### Missing Values — Explanation & Decision

| Column | Missing | Root cause | Decision |
|---|---|---|---|
| `review_score` | 646 | Customer did not submit a review | Expected — `has_review` flag added below |
| `customer_lat/lng` | ~265 | Customer zip not in geolocation table | Deferred to Feature Engineering — state-level fill would distort distance calculations |
| `seller_lat/lng` | ~175 | Seller zip not in geolocation table | Same — deferred |
| `avg_product_weight_g` | 16 | 2 product_ids exist in items but not in products table | Fill with dataset median (16 rows, <0.02% impact) |
| `avg_product_volume_cm3` | 16 | Same 2 products | Fill with dataset median |
| `product_category` | ~1,351 | Products with `category = 'unknown'` have no English translation | Fill with `'unknown'` |


 **Why not fill lat/lng with state-level median?**  

 Brazil's states are geographically enormous.
 Filling a missing Rio de Janeiro
 coordinate with the state centroid could place a customer hundreds of kilometres
 from their actual location, producing invalid distance features.
 The correct strategy (city-level or zip-level interpolation) will be decided
 after EDA reveals how many distance features are affected.


In [12]:
# ── Fix 1: product_category — fill 'unknown' (translation gap) ───────────────
df['product_category'] = df['product_category'].fillna('unknown')
print(f"product_category  missing after fix : {df['product_category'].isnull().sum()} ✓")

# ── Fix 2: has_review binary flag ─────────────────────────────────────────────
# WHY: Instead of guessing review_score for 646 orders, we add a binary flag
# that lets the model learn from the absence of a review as a signal.
df['has_review'] = df['review_score'].notna().astype(int)
print(f"has_review: {df['has_review'].value_counts().to_dict()}")
print(f"  → {df['has_review'].sum():,} orders have a review ({df['has_review'].mean()*100:.1f}%)")

# ── Fix 3: product weight & volume — fill 16 rows with dataset median ─────────
# WHY: Only 16 rows affected (2 orphaned product_ids). Median is appropriate
# here because the sample size is too small for category-level imputation.
median_weight = df['avg_product_weight_g'].median()
median_volume = df['avg_product_volume_cm3'].median()

df['avg_product_weight_g']   = df['avg_product_weight_g'].fillna(median_weight)
df['avg_product_volume_cm3'] = df['avg_product_volume_cm3'].fillna(median_volume)
print(f"avg_product_weight_g   missing after fix : {df['avg_product_weight_g'].isnull().sum()} ✓")
print(f"avg_product_volume_cm3 missing after fix : {df['avg_product_volume_cm3'].isnull().sum()} ✓")

# ── Fix 4: payment columns — 1 order has no payment record ────────────────────
# pandas cast int columns to float64 because int cannot hold NaN.
# We fill with 0 and restore int dtype. The order is kept because
# it has valid delivery data and a valid is_late value.
df['n_payment_methods']   = df['n_payment_methods'].fillna(0).astype(int)
df['max_installments']    = df['max_installments'].fillna(0).astype(int)
df['total_payment_value'] = df['total_payment_value'].fillna(0.0)
df['payment_type']        = df['payment_type'].fillna('unknown')
print(f"Payment columns fixed — dtypes: n_payment_methods={df['n_payment_methods'].dtype}, max_installments={df['max_installments'].dtype}")

# ── Final missing value check ─────────────────────────────────────────────────
remaining = df.isnull().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print(f"\nRemaining missing values (expected: only lat/lng + review_score):")
print(remaining.to_string() if len(remaining) > 0 else "  None ✓")


product_category  missing after fix : 0 ✓
has_review: {1: 95815, 0: 646}
  → 95,815 orders have a review (99.3%)
avg_product_weight_g   missing after fix : 0 ✓
avg_product_volume_cm3 missing after fix : 0 ✓
Payment columns fixed — dtypes: n_payment_methods=int32, max_installments=int32

Remaining missing values (expected: only lat/lng + review_score):
review_score           646
customer_lat           265
customer_lng           265
customer_geo_points    265
seller_lat             215
seller_lng             215
seller_geo_points      215


---

## Unique Values Inspection & Final Fixes

Quick scan of all columns. Any column with ≤ 10 unique values gets
a full value count — useful for catching unexpected categories or flags.


In [13]:
# ── Unique values scan ────────────────────────────────────────────────────────
print("Columns with ≤ 10 unique values:")
for col in df.columns:
    n_unique = df[col].nunique()
    if n_unique <= 10:
        print(f"\n  {col} — {n_unique} unique values")
        print(df[col].value_counts().to_string())


Columns with ≤ 10 unique values:

  order_status — 2 unique values
order_status
delivered    96455
canceled         6

  is_late — 2 unique values
is_late
0    88635
1     7826

  is_delivered — 2 unique values
is_delivered
1    96455
0        6

  delivery_status — 2 unique values
delivery_status
early    88635
late      7826

  n_unique_sellers — 5 unique values
n_unique_sellers
1    95186
2     1216
3       54
4        3
5        2

  payment_type — 5 unique values
payment_type
credit_card    73939
boleto         19177
voucher         1861
debit_card      1483
unknown            1

  review_score — 5 unique values
review_score
5.0    56749
4.0    18889
1.0     9347
3.0     7908
2.0     2922

  has_review — 2 unique values
has_review
1    95815
0      646


### Finding: 6 Canceled Orders

**Issue:** 6 orders with `order_status = 'canceled'` survived into the dataset.

**Root cause:** Notebook 01 filtered on missing timestamps — these 6 orders
have a `order_delivered_customer_date` populated (data inconsistency in the
raw source data), so they were not caught by the missing-value filter.

**Decision:** Remove them. A canceled order cannot have a valid delivery event.

**Impact:** 96,461 → **96,455** final rows.


In [14]:
# ── Inspect canceled orders ───────────────────────────────────────────────────
canceled = df[df['order_status'] == 'canceled']
print(f"Canceled orders found: {len(canceled)}")
if len(canceled) > 0:
    print(canceled[['order_id', 'order_status', 'is_late', 'delivery_days',
                     'order_delivered_customer_date']].to_string())

# ── Remove canceled orders ────────────────────────────────────────────────────
df = df[df['order_status'] != 'canceled'].copy()
print(f"\nRows after removing canceled: {len(df):,}  (96,461 → {len(df):,})")


Canceled orders found: 6
                               order_id order_status  is_late  delivery_days order_delivered_customer_date
2843   1950d777989f6a877539f53795b4c3c3     canceled        1             30           2018-03-21 22:03:51
8545   dabf2b0e35b423f94618bf965fcb7514     canceled        0              7           2016-10-16 14:36:59
56529  770d331c84e5b214bd9dc70a10b829d0     canceled        0              7           2016-10-14 15:07:11
57557  8beb59392e21af5eb9547ae1a9938d06     canceled        0             10           2016-10-19 18:47:43
89861  65d1e226dfaeb8cdc42f665422522d14     canceled        0             35           2016-11-08 10:58:34
91573  2c45c33d2f9cb8ff8b1c86cc28c11c30     canceled        0             30           2016-11-09 14:53:50

Rows after removing canceled: 96,455  (96,461 → 96,455)


### Finding: High n_payment_methods (max = 26)

28 orders show more than 10 payment methods — all are `voucher` type.
Each voucher/coupon is recorded as a separate `payment_sequential` entry.
An order with 26 vouchers used 26 different coupons. **This is not a data error.**


In [15]:
# ── Inspect high payment methods ──────────────────────────────────────────────
high_pay = df[df['n_payment_methods'] > 10]
print(f"Orders with > 10 payment methods : {len(high_pay)}")
if len(high_pay) > 0:
    print(high_pay[['order_id', 'n_payment_methods', 'total_payment_value', 'payment_type']].to_string())
print("\n→ All are 'voucher' type. Multiple vouchers per order is expected behavior in Olist.")


Orders with > 10 payment methods : 28
                               order_id  n_payment_methods  total_payment_value payment_type
7474   285c2e15bebd4ac83635ccc563dc71f4                 22                40.85      voucher
9136   27a940efdd448db29463b53ea0cfa2f4                 11                87.66      voucher
9143   4069c489933782af79afcd3a0e4d693c                 11               147.15      voucher
16865  21577126c19bf11a0b91592e5844ba78                 15                86.99      voucher
17046  654da57158d96035814657b5143bb11b                 11               317.80      voucher
17833  1a611328643ae11146ba09a4425d2e12                 12                69.97      voucher
22256  4bfcba9e084f46c8e3cb49b0fa6e6159                 15               740.76      voucher
26264  fedcd9f7ccdc8cba3a18defedd1a5547                 19               205.74      voucher
27622  67d83bd36ec2c7fb557742fb58837659                 12                72.17      voucher
29249  0bbb3f7791a87d0307555e57d

---

## Final Dataset Inspection

Last look before saving. Verify shape, dtypes, and distributions.


In [16]:
# ── Shape ─────────────────────────────────────────────────────────────────────
print(f"Final shape: {df.shape}")
print(f"  Rows    : {len(df):,}  (expected: 96,455)")
print(f"  Columns : {df.shape[1]}")


Final shape: (96455, 37)
  Rows    : 96,455  (expected: 96,455)
  Columns : 37


In [17]:
# Sample rows
df.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,is_delivered,...,max_installments,payment_type,review_score,customer_lat,customer_lng,customer_geo_points,seller_lat,seller_lng,seller_geo_points,has_review
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,1,...,1,voucher,4.0,-23.576983,-46.587161,24.0,-23.680729,-46.444238,207.0,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,1,...,1,boleto,4.0,-12.177924,-44.660711,19.0,-19.807681,-43.980427,71.0,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,1,...,3,credit_card,5.0,-16.745150,-48.514783,25.0,-21.363502,-48.229601,170.0,1


In [18]:
# Data types
df.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
is_late                                   int64
is_delivered                              int64
delivery_status                          object
delivery_days                             int64
customer_unique_id                       object
customer_zip_code_prefix                  int64
customer_state                           object
customer_city                            object
n_items                                   int64
n_unique_sellers                          int64
total_price                             float64
total_freight                           float64
avg_product_weight_g                    

In [19]:
# Statistical summary
df.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,is_delivered,delivery_days,customer_zip_code_prefix,n_items,...,n_payment_methods,max_installments,review_score,customer_lat,customer_lng,customer_geo_points,seller_lat,seller_lng,seller_geo_points,has_review
count,96455,96455,96455,96455,96455,96455.000000,96455.0,96455.000000,96455.000000,96455.000000,...,96455.000000,96455.000000,95809.000000,96190.000000,96190.000000,96190.000000,96240.000000,96240.000000,96240.000000,96455.000000
mean,2018-01-02 00:26:23.673806336,2018-01-02 10:43:03.401710592,2018-01-05 05:53:41.200134656,2018-01-14 13:49:28.328526336,2018-01-25 18:04:03.421284864,0.081126,1.0,12.093100,35199.156674,1.142222,...,1.045161,2.928309,4.156227,-21.204880,-46.190295,152.305198,-22.794460,-47.224089,144.134965,0.993303
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,0.000000,1.0,0.000000,1003.000000,1.000000,...,0.000000,0.000000,1.000000,-33.689948,-72.668881,1.000000,-32.079231,-63.893565,1.000000,0.000000
25%,2017-09-14 09:39:02.500000,2017-09-14 14:42:23,2017-09-18 17:06:18.500000,2017-09-25 22:56:47,2017-10-05 00:00:00,0.000000,1.0,6.000000,11355.000000,1.000000,...,1.000000,1.000000,4.000000,-23.589853,-48.119196,53.000000,-23.612734,-48.807104,55.000000,1.000000
50%,2018-01-20 20:00:12,2018-01-22 13:49:24,2018-01-24 16:28:58,2018-02-02 19:52:30,2018-02-16 00:00:00,0.000000,1.0,10.000000,24436.000000,1.000000,...,1.000000,2.000000,5.000000,-22.924970,-46.632292,103.000000,-23.425556,-46.747854,108.000000,1.000000
75%,2018-05-05 18:53:33,2018-05-06 10:52:57.500000,2018-05-08 14:34:30,2018-05-15 23:09:15.500000,2018-05-28 00:00:00,0.000000,1.0,15.000000,59056.000000,1.000000,...,1.000000,4.000000,5.000000,-20.140216,-43.627314,195.000000,-21.757321,-46.518679,207.000000,1.000000
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,1.000000,1.0,209.000000,99980.000000,21.000000,...,26.000000,24.000000,5.000000,3.842508,-34.799347,1146.000000,-2.501242,-34.855616,965.000000,1.000000
std,NaN,NaN,NaN,NaN,NaN,0.273030,0.0,9.551209,29839.839947,0.538857,...,0.370710,2.712909,1.284557,5.589107,4.049687,152.343085,2.757186,2.350421,118.962814,0.081564


**Key observations from the statistical summary:**

- `delivery_days` ranges 0 → ~209 — outliers retained intentionally (documented in NB01)
- `customer_lat/lng` + `seller_lat/lng` — ~99.7% coverage, sufficient for distance features
- `customer_geo_points` / `seller_geo_points` — records how many GPS readings backed each coordinate
- `review_score` — 646 NaN expected (orders without a review)
- `is_late` — 8.1% positive class — class imbalance to handle during modeling


---

## Save Dataset

We save the final dataset as a CSV. The file is named `olist_merged.csv`
and is ready for the EDA notebook.

**What makes this dataset "high-resolution":**
- GPS coordinates backed by the **mean of all valid readings** per zip code
  (not a single median centroid)
- `*_geo_points` confidence columns included for spatial analysis
- All flags from NB01 preserved (`is_late`, `is_delivered`, `delivery_status`,
  `is_valid_city`, `is_outside_brazil`)
- No statistical manipulation of missing values beyond the 16 documented rows


In [20]:
# ── Save the dataset ─────────────────────────────────────────
import os
os.makedirs('../data/processed/', exist_ok=True)

output_path = '../data/processed/olist_merged.csv'
df.to_csv(output_path, index=False)

print("=" * 60)
print("  Dataset Saved Successfully")
print("=" * 60)
print(f"  Path   : {output_path}")
print(f"  Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Size   : {os.path.getsize(output_path)/1024/1024:.1f} MB  (approx)")
print()
print(f"  Late orders  : {df['is_late'].sum():,}  ({df['is_late'].mean()*100:.1f}%)")
print(f"  With review  : {df['has_review'].sum():,}  ({df['has_review'].mean()*100:.1f}%)")
print(f"  With GPS     : {df['customer_lat'].notna().sum():,} customer  /  {df['seller_lat'].notna().sum():,} seller")
print()
print("  ✓ Ready for EDA notebook (03_eda.ipynb)")
print("=" * 60)


  Dataset Saved Successfully
  Path   : ../data/processed/olist_merged.csv
  Shape  : 96,455 rows × 37 columns
  Size   : 36.9 MB  (approx)

  Late orders  : 7,825  (8.1%)
  With review  : 95,809  (99.3%)
  With GPS     : 96,190 customer  /  96,240 seller

  ✓ Ready for EDA notebook (03_eda.ipynb)


---

## Merging Summary

**Input:** 9 cleaned tables from Notebook 01  
**Output:** `olist_merged.csv` — **96,455 rows × 35 columns**

---

### Complete Join Log

| Step | Operation | Join key | Rows before | Rows after | Notes |
|---|---|---|---|---|---|
| 0 | Load orders (anchor) | — | — | 96,461 | Base table |
| 1 | LEFT JOIN customers | customer_id | 96,461 | 96,461 | Adds zip, state, city |
| 2 | LEFT JOIN items_agg | order_id | 96,461 | 96,461 | Aggregated from 112,650 items |
| 3 | LEFT JOIN payments_agg | order_id | 96,461 | 96,461 | Aggregated from 103,886 payments |
| 4 | LEFT JOIN reviews_clean | order_id | 96,461 | 96,461 | De-duped from 99,224 reviews |
| 5 | LEFT JOIN customer_geo | customer_zip_code_prefix | 96,461 | 96,461 | Mean of all GPS per zip |
| 6 | LEFT JOIN seller_geo | seller_zip_code_prefix | 96,461 | 96,461 | Mean of all GPS per zip |
| 7 | Drop customer_id | — | — | — | Replaced by customer_unique_id |
| 8 | Fix: product_category fillna | — | — | — | 1,351 → 'unknown' |
| 9 | Add has_review flag | — | — | — | 646 NaN review_score |
| 10 | Fix: weight/volume median fill | — | — | — | 16 rows, 2 orphaned products |
| 11 | Fix: payment column dtypes | — | — | — | 1 order missing payment |
| 12 | Remove 6 canceled orders | — | 96,461 | **96,455** | Status/delivery inconsistency |

---

### Geolocation Methodology

| | Old approach | **New approach** |
|---|---|---|
| Method | Median per zip code | **Mean of all valid GPS readings** |
| Input rows used | ~1,000,163 (then discarded) | ~1,000,163 (aggregated, not discarded) |
| Output per zip | 1 centroid | 1 weighted mean point |
| Confidence metric | None | `*_geo_points` column |
| Spatial accuracy | Lower — centroid bias | Higher — full GPS cloud |

---

### Final Column Inventory

| Column group | Columns | Source |
|---|---|---|
| Order identifiers | order_id, customer_unique_id, order_status | orders |
| Timestamps | order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date | orders |
| Delivery flags | is_late, is_delivered, delivery_status, delivery_days | NB01 + NB02 |
| Customer location | customer_zip_code_prefix, customer_state, customer_city, customer_lat, customer_lng, customer_geo_points | customers + geolocation |
| Items summary | n_items, n_unique_sellers, total_price, total_freight, avg_product_weight_g, avg_product_volume_cm3, product_category | items_agg |
| Seller location | seller_zip_code_prefix, seller_state, seller_lat, seller_lng, seller_geo_points | sellers + geolocation |
| Payment summary | total_payment_value, n_payment_methods, max_installments, payment_type | payments_agg |
| Review | review_score, has_review | reviews |

---

### Missing Values in Final Dataset

| Column | Missing | Reason | Status |
|---|---|---|---|
| `review_score` | 646 | No review submitted | Deferred — `has_review` flag created |
| `customer_lat` / `customer_lng` | ~265 | Zip not in geolocation table | Deferred to Feature Engineering |
| `seller_lat` / `seller_lng` | ~175 | Zip not in geolocation table | Deferred to Feature Engineering |
| All other columns | 0 | — | ✓ Fully resolved |
